### Conversión de CSV a Parquet

Con los datos descargados, el siguiente paso es convertirlos del formato CSV comprimido (`.csv.gz`) a **Parquet**, un formato columnar binario mucho más eficiente para su procesamiento con Spark.

#### Creación de la sesión Spark

El primer paso es importar PySpark y crear una sesión Spark. La dirección del nodo maestro se lee de la variable de entorno `SPARK_MASTER`, lo que permite que el cuaderno funcione tanto en local como apuntando a un clúster real sin modificar el código. `appName` es simplemente un nombre identificativo que aparecerá en la interfaz web de Spark.

In [1]:
import pyspark
import os
from pyspark.sql import SparkSession

# Creamos una sesión PySpark contra nuestro servicio "spark-master"
spark = SparkSession.builder \
    .master(os.environ.get('SPARK_MASTER')) \
    .appName("csv-to-parquet") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 19:35:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Importación del módulo de tipos

Se importa el módulo `types` de PySpark, que proporciona todas las clases necesarias para definir esquemas de datos explícitos: `StructType`, `StructField`, `IntegerType`, `DoubleType`, `StringType`, `TimestampType`, etc.

In [2]:
from pyspark.sql import types

#### Esquema de los taxis verdes

Se define el esquema explícito para los ficheros de taxis verdes. Cada `StructField` recibe tres argumentos:

* El nombre de la columna, que debe coincidir exactamente con la cabecera del CSV.
* El tipo de dato (`IntegerType`, `DoubleType`, `StringType`, `TimestampType`...).
* Un booleano que indica si el campo puede ser nulo (`True` en todos los casos aquí).

Definir el esquema manualmente en lugar de dejar que Spark lo infiera automáticamente tiene varias ventajas:

* Spark no necesita leer el fichero dos veces (una para inferir tipos y otra para cargar datos).
* Se evitan inferencias incorrectas, especialmente en columnas de fechas y horas que Spark suele cargar como texto.
* El código documenta explícitamente la estructura esperada de los datos.

In [3]:
green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("ehail_fee", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("trip_type", types.IntegerType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

#### Esquema de los taxis amarillos

Esquema análogo al anterior pero para los ficheros de taxis amarillos.

> Los nombres de las columnas de fecha difieren: los taxis verdes usan el prefijo `lpep_` (_Lincoln Park Express_) mientras que los amarillos usan `tpep_` (_Taxi Passenger Enhancement Program_).
> El orden de los campos también varía entre los dos tipos de taxi, lo que refuerza la necesidad de definir los esquemas por separado en lugar de reutilizar uno solo.

In [4]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

#### Función de conversión

Esta función centraliza toda la lógica de conversión y acepta el tipo de taxi, el año y el esquema como parámetros. Su funcionamiento paso a paso es:

* **Bucle de meses**: itera del 1 al 12, formateando el mes con dos dígitos (`{month:02d}`) para construir el nombre de fichero correcto.
* **Comprobación de existencia**: si el fichero de entrada no existe (por ejemplo, porque ese mes no está disponible), lo avisa y continúa. Esto hace la función tolerante a huecos en los datos.
* **Lectura del CSV**: carga el fichero comprimido con `spark.read.csv`. Se activa `header=true` para que Spark use la primera fila como nombres de columnas, y se aplica el esquema definido previamente.
* **Repartición**: antes de escribir, se redistribuye el DataFrame en 4 particiones con `.repartition(4)`. Esto controla cuántos ficheros Parquet de salida se generan por mes, evitando tanto el problema de tener un único fichero enorme como el de tener cientos de ficheros diminutos.
* **Escritura en Parquet**: escribe el resultado en el directorio de salida organizado como `/data/parquet/{tipo}/{año}/{mes}/`.

In [13]:
import os

def convert_to_parquet(taxi_type, year, schema):
    for month in range(1, 13):
        print(f"Procesando: {taxi_type} - {year}/{month}")

        input_path = f"/data/raw/{taxi_type}/{taxi_type}_tripdata_{year}-{month:02d}.csv.gz"
        output_path = f"/data/parquet/{taxi_type}/{year}/{month:02d}"

        if not os.path.exists(input_path):
            print(f"> El fichero {year}/{month} no existe y será ignorado")
            continue

        df = spark.read \
            .option("header", "true") \
            .schema(schema) \
            .csv(input_path)

        df \
            .repartition(4) \
            .write.parquet(output_path)

[Stage 136:>                                                        (0 + 1) / 1]

El resultado queda organizado así:

```
data/
└── parquet/
    ├── green/
    │   ├── 2019/
    │   │   ├── 01/
    │   │   │   ├── part-00000-....parquet
    │   │   │   ├── part-00001-....parquet
    │   │   │   ├── part-00002-....parquet
    │   │   │   └── part-00003-....parquet
    │   │   ├── 02/
    │   │   └── ...
    │   └── ...
    └── yellow/
        └── ...
```

#### Ejecución de la conversión

Se invoca la función para los tres años disponibles y para cada tipo de taxi. Gracias a que la lógica está encapsulada en una función parametrizada, añadir más años o tipos de taxi es trivial. El proceso es computacionalmente intensivo ya que Spark está leyendo, descomprimiendo y reescribiendo varios GB de datos, pero solo necesita ejecutarse una vez: una vez que los Parquet están en disco, todas las operaciones posteriores usarán esos ficheros directamente.

In [ ]:
# Procesamos los ficheros de taxis verdes
convert_to_parquet('green', 2019, green_schema)
convert_to_parquet('green', 2020, green_schema)
convert_to_parquet('green', 2021, green_schema)

# Procesamos los ficheros de taxis amarillos
convert_to_parquet('yellow', 2019, yellow_schema)
convert_to_parquet('yellow', 2020, yellow_schema)
convert_to_parquet('yellow', 2021, yellow_schema)

Procesando: green - 2019/1


Procesando: green - 2019/2


Procesando: green - 2019/3


Procesando: green - 2019/4


Procesando: green - 2019/5


...

Procesando: green - 2021/6


Procesando: green - 2021/7


Procesando: green - 2021/8
> El fichero 2021/8 no existe y será ignorado
Procesando: green - 2021/9
> El fichero 2021/9 no existe y será ignorado
Procesando: green - 2021/10
> El fichero 2021/10 no existe y será ignorado
Procesando: green - 2021/11
> El fichero 2021/11 no existe y será ignorado
Procesando: green - 2021/12
> El fichero 2021/12 no existe y será ignorado
Procesando: yellow - 2019/1


Procesando: yellow - 2019/2


[Stage 326:>                                                        (0 + 1) / 1]